# Диагностика времени до успешной встречи (`success_timing_diagnostics`)

Расширение robustness / diagnostics к основному empirical pipeline.
**Не изменяет** основную идентификационную стратегию и определения `change_type` / treatment.

## Исследовательский вопрос

Могли ли изменения конфигурации зон доставки повлиять на **скорость** достижения
успешной встречи, даже если итоговая вероятность `success_flg` / `success_within_H`
существенно не изменилась?

## Диагностические гипотезы

**H0.** После изменения зоны доставки распределение времени от заявки до успешной
встречи не изменилось относительно соответствующей контрольной / дореформенной динамики.

**H1.** Изменение зоны доставки связано со сдвигом времени до successful meeting:
успешные встречи происходят быстрее или медленнее.

## Важное методологическое ограничение

Анализ `days_to_success` **только среди** `success_flg == 1` является
**CONDITIONED-ON-OUTCOME** диагностикой и **не должен** интерпретироваться как
causal effect: treatment потенциально влияет на вероятность попасть в
`success_flg == 1`.

Поэтому основной акцент notebook — на **cumulative success outcomes среди всех
достаточно зрелых заявок** (`success_within_H` с explicit maturity / right-censoring),
а conditional time-to-success остаётся вспомогательной descriptive диагностикой.

## Согласованность с основным pipeline

- `build_analysis_panel` (`min_orders_per_hex=5`, исключение когорты `2022-10-19`)
- CORE `change_type`: `region_only`, `workmode_only`, `region_and_workmode`
- для success-исходов дополнительно исключается когорта `2022-07-27`
- горизонты через `last_mile.outcomes.add_success_within_horizon` /
  `success_horizon_coverage`
- FE event-study: hex FE + calendar-day FE, clustered SE by hex,
  reference week = `-1`, окно примерно `-8…+8`


In [1]:
# ============================================================
# 0. Пути и директории вывода
# ============================================================

from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
if cwd.name.lower() in {"notebooks", "notebook"}:
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_RAW = PROJECT_ROOT / "data" / "raw"
APPLICATIONS_PATH = DATA_RAW / "application_dataset.csv"
HEXAGONS_PATH = DATA_RAW / "hexagons_dataset.csv"

OUT_DIR = PROJECT_ROOT / "outputs" / "success_timing_diagnostics"
FIG_DIR = PROJECT_ROOT / "figures" / "success_timing_diagnostics"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT: <repository root>")
print("OUT_DIR:", OUT_DIR.relative_to(PROJECT_ROOT).as_posix())
print("FIG_DIR:", FIG_DIR.relative_to(PROJECT_ROOT).as_posix())

assert APPLICATIONS_PATH.exists(), f"Не найден файл: {APPLICATIONS_PATH}"
assert HEXAGONS_PATH.exists(), f"Не найден файл: {HEXAGONS_PATH}"


PROJECT_ROOT: <repository root>
OUT_DIR: outputs/success_timing_diagnostics
FIG_DIR: figures/success_timing_diagnostics


In [2]:
# ============================================================
# 1. Импорты и стиль графиков
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

candidate_paths = [
    PROJECT_ROOT,
    PROJECT_ROOT / "last_mile",
    PROJECT_ROOT / "src",
    Path.cwd(),
    Path.cwd().parent,
]
for p in candidate_paths:
    if p.exists():
        sys.path.insert(0, str(p))

from last_mile.filter import (
    build_analysis_panel,
    classify_hexagons,
    resolve_hexagon_treatments,
    CORE_CHANGE_TYPES,
    STUDY_START,
    STUDY_END,
    EXCLUDED_COHORTS,
)
from last_mile.outcomes import (
    add_success_within_horizon,
    success_horizon_coverage,
)
from last_mile.plot_style import (
    PALETTE,
    save_figure,
    style_axes,
    with_plot_style,
    plot_style,
)

try:
    from linearmodels.panel import PanelOLS
except ImportError as e:
    raise ImportError(
        "Нужен пакет linearmodels. Установите: pip install linearmodels"
    ) from e

RNG = np.random.default_rng(42)
N_BOOTSTRAP = 200  # можно увеличить позже

TYPE_ORDER = ["region_and_workmode", "region_only", "workmode_only"]
assert set(CORE_CHANGE_TYPES) == set(TYPE_ORDER)

TYPE_LABELS_EN = {
    "region_and_workmode": "Region + work mode",
    "region_only": "Region only",
    "workmode_only": "Work mode only",
}
TYPE_COLORS = {
    "region_and_workmode": PALETTE["region_and_workmode"],
    "region_only": PALETTE["region_only"],
    "workmode_only": PALETTE["workmode_only"],
}

REF_DAYS = [3, 7, 14, 25]
HORIZONS_H = [1, 3, 5, 7, 10, 14, 20, 25]
CURVE_DAYS = list(range(0, 26))
DID_HORIZONS = [3, 7, 14, 25]
EVENT_WEEK_MIN, EVENT_WEEK_MAX = -8, 8
SUCCESS_OBSERVATION_END = pd.Timestamp("2022-10-18")
EXCLUDED_COHORT_FOR_CONVERSION = pd.Timestamp("2022-07-27")

print("CORE_CHANGE_TYPES:", CORE_CHANGE_TYPES)
print("STUDY_START / STUDY_END:", STUDY_START.date(), STUDY_END.date())
print("EXCLUDED_COHORTS:", sorted(str(x.date()) for x in EXCLUDED_COHORTS))
print("N_BOOTSTRAP:", N_BOOTSTRAP)


CORE_CHANGE_TYPES: ['region_only', 'workmode_only', 'region_and_workmode']
STUDY_START / STUDY_END: 2022-04-01 2022-10-18
EXCLUDED_COHORTS: ['2022-10-19']
N_BOOTSTRAP: 200


## 1. Загрузка и согласование с основным pipeline

Используем `build_analysis_panel` и `success_horizon_coverage` без копирования
логики `change_type` / multi-treatment resolution.


In [3]:
# ============================================================
# 2. Загрузка панели (тот же pipeline, что final_empirical_recalculation)
# ============================================================

result = build_analysis_panel(
    applications_path=APPLICATIONS_PATH,
    hexagons_path=HEXAGONS_PATH,
    min_orders_per_hex=5,
)

panel = result["panel"].copy()
apps_t = result["treated_orders"].copy()
apps_c = result["control_orders"].copy()
cohort_map = result["cohort_map"].copy()

assert cohort_map["hex"].is_unique, "cohort_map['hex'] должен быть уникален"

for df in (apps_t, apps_c):
    if "cohort" in df.columns and "treatment_date" not in df.columns:
        df["treatment_date"] = df["cohort"]
    if "date" not in df.columns:
        df["date"] = df["request_timestamp"].dt.normalize()
    for col in ("request_timestamp", "first_success_dttm", "treatment_date"):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

if "days_from_treatment" not in apps_t.columns:
    apps_t["days_from_treatment"] = (
        apps_t["request_timestamp"] - apps_t["treatment_date"]
    ).dt.days

# Sanity: single treatment date per treated hex
assert (
    apps_t.groupby("hex")["treatment_date"].nunique().le(1).all()
), "У treated hex должна быть одна treatment_date"

SUCCESS_HORIZONS = tuple(sorted(set(HORIZONS_H + DID_HORIZONS + [20])))
apps_t, success_coverage_t = success_horizon_coverage(
    apps_t,
    horizons=SUCCESS_HORIZONS,
    observation_end=SUCCESS_OBSERVATION_END,
)
apps_c, success_coverage_c = success_horizon_coverage(
    apps_c,
    horizons=SUCCESS_HORIZONS,
    observation_end=SUCCESS_OBSERVATION_END,
)
success_coverage_t["sample"] = "treated"
success_coverage_c["sample"] = "never_treated"
success_horizon_coverage_table = pd.concat(
    [success_coverage_t, success_coverage_c], ignore_index=True
)

# Assert immature outcomes are NA
for H in SUCCESS_HORIZONS:
    mcol = f"mature_success_{H}"
    ocol = f"success_within_{H}"
    assert apps_t.loc[~apps_t[mcol], ocol].isna().all()
    assert apps_c.loc[~apps_c[mcol], ocol].isna().all()

OBS_MIN = min(apps_t["request_timestamp"].min(), apps_c["request_timestamp"].min())
OBS_MAX = max(apps_t["request_timestamp"].max(), apps_c["request_timestamp"].max())

print("OBS_MIN:", OBS_MIN)
print("OBS_MAX:", OBS_MAX)
print("treated orders:", f"{len(apps_t):,}")
print("control orders:", f"{len(apps_c):,}")
display(success_horizon_coverage_table)


[INFO] resolve_hexagon_treatments: исключено 15832 multi-treatment гексагонов; сохранено 3613240 single/never.
[INFO] multi-treatment diagnostics: <PROJECT_ROOT>/…
[INFO] attach_hex_metadata: отброшено 109 заявок из гексагонов, не представленных в hexagons_dataset.
[INFO] exclude_cohorts: исключено 683082 заявок из когорт ['2022-10-19'].
[INFO] build_analysis_panel: после порогового фильтра (≥5 заявок) осталось 13255 уникальных гексагонов.

СВОДКА АНАЛИТИЧЕСКОЙ ПАНЕЛИ
  Период:              2022-04-01 - 2022-10-18
  Наблюдений (hex×day):   302,398
  Гексагонов всего:        13,255
  - трактуемых:             9,327
  - never-treated:          3,928

  Когорты (включённые):
    2022-07-27  →  437 гексагонов
    2022-08-12  →  1358 гексагонов
    2022-08-19  →  2723 гексагонов
    2022-09-22  →  4809 гексагонов

  Исключённые когорты: ['2022-10-19']

OBS_MIN: 2022-04-01 00:00:00
OBS_MAX: 2022-10-18 00:00:00
treated orders: 781,369
control orders: 323,125


,horizon_days,observation_end,n_requests_total,n_requests_mature,mature_share,n_success_within_horizon,sample
0,1,2022-10-18,781369,779777,0.997963,55976,treated
1,3,2022-10-18,781369,769688,0.985051,271974,treated
2,5,2022-10-18,781369,757544,0.969509,371016,treated
3,7,2022-10-18,781369,745320,0.953864,409134,treated
4,10,2022-10-18,781369,727828,0.931478,420663,treated
5,14,2022-10-18,781369,707762,0.905797,419640,treated
6,20,2022-10-18,781369,688025,0.880538,414227,treated
7,25,2022-10-18,781369,672005,0.860035,406590,treated
8,1,2022-10-18,323125,322650,0.998530,28016,never_treated
9,3,2022-10-18,323125,318660,0.986182,118158,never_treated


In [4]:
# ============================================================
# 3. Когорты и аналитический frame (CORE + exclusion 2022-07-27)
# ============================================================

VALID_COHORTS_ALL = {
    pd.Timestamp(x) for x in pd.to_datetime(apps_t["treatment_date"].dropna().unique())
}
VALID_COHORTS_CONVERSION = VALID_COHORTS_ALL - {EXCLUDED_COHORT_FOR_CONVERSION}


def cohorts_for_outcome(outcome: str) -> set[pd.Timestamp]:
    # Совпадает с final_empirical_recalculation: success_within_* без 2022-07-27
    if outcome.startswith("success_within_"):
        return VALID_COHORTS_CONVERSION
    return VALID_COHORTS_ALL


apps_all = pd.concat([apps_t, apps_c], ignore_index=True)

# CORE analysis frame: treated CORE + never-treated control
core_mask = apps_all["change_type"].isin(TYPE_ORDER) | (~apps_all["is_treated"])
apps_core_raw = apps_all.loc[core_mask].copy()

# Success-timing analysis excludes 2022-07-27 treated cohort (как для success_within_*)
apps_core = apps_core_raw[
    (~apps_core_raw["is_treated"])
    | apps_core_raw["treatment_date"].isin(VALID_COHORTS_CONVERSION)
].copy()

apps_core["is_post"] = (
    apps_core["is_treated"]
    & apps_core["treatment_date"].notna()
    & (apps_core["request_timestamp"].dt.normalize() >= apps_core["treatment_date"])
)
apps_core["period"] = np.where(
    ~apps_core["is_treated"],
    "control",
    np.where(apps_core["is_post"], "post", "pre"),
)

print("VALID_COHORTS_ALL:", sorted(str(c.date()) for c in VALID_COHORTS_ALL))
print(
    "VALID_COHORTS_CONVERSION (success):",
    sorted(str(c.date()) for c in VALID_COHORTS_CONVERSION),
)
print(
    f"apps_core: {len(apps_core):,} | treated CORE hex: "
    f"{apps_core.loc[apps_core['change_type'].isin(TYPE_ORDER), 'hex'].nunique()} | "
    f"control hex: {apps_core.loc[~apps_core['is_treated'], 'hex'].nunique()}"
)

composition = (
    apps_core.loc[apps_core["change_type"].isin(TYPE_ORDER)]
    .drop_duplicates("hex")
    .pivot_table(
        index="treatment_date",
        columns="change_type",
        values="hex",
        aggfunc="nunique",
        fill_value=0,
    )
    .reindex(columns=TYPE_ORDER, fill_value=0)
    .astype(int)
)
display(composition)


VALID_COHORTS_ALL: ['2022-07-27', '2022-08-12', '2022-08-19', '2022-09-22']
VALID_COHORTS_CONVERSION (success): ['2022-08-12', '2022-08-19', '2022-09-22']
apps_core: 515,984 | treated CORE hex: 6991 | control hex: 3928


change_type,region_and_workmode,region_only,workmode_only
treatment_date,,,
2022-08-12,370,189,68
2022-08-19,350,1,806
2022-09-22,1502,340,751


## 2. Data quality

Сначала показываем число подозрительных случаев, затем явно задаём правила фильтрации.
Подозрительные записи **не** исправляются silently.


In [5]:
# ============================================================
# 4. Data quality checks + explicit filters
# ============================================================

dq = apps_core.copy()
dq["success_flg"] = pd.to_numeric(dq["success_flg"], errors="coerce")

has_success_ts = dq["first_success_dttm"].notna()
success1 = dq["success_flg"] == 1
success0 = dq["success_flg"] == 0

delay_hours_raw = (
    (dq["first_success_dttm"] - dq["request_timestamp"]).dt.total_seconds() / 3600.0
)
dq["success_delay_hours_raw"] = delay_hours_raw
dq["success_delay_days_raw"] = delay_hours_raw / 24.0

quality_rows = [
    {"check": "n_applications", "n": int(len(dq)), "share": 1.0},
    {
        "check": "success_flg_eq_1",
        "n": int(success1.sum()),
        "share": float(success1.mean()),
    },
    {
        "check": "success_flg_eq_0",
        "n": int(success0.sum()),
        "share": float(success0.mean()),
    },
    {
        "check": "success_flg_missing",
        "n": int(dq["success_flg"].isna().sum()),
        "share": float(dq["success_flg"].isna().mean()),
    },
    {
        "check": "success1_missing_first_success_dttm",
        "n": int((success1 & ~has_success_ts).sum()),
        "share": float((success1 & ~has_success_ts).mean()),
    },
    {
        "check": "success0_has_first_success_dttm",
        "n": int((success0 & has_success_ts).sum()),
        "share": float((success0 & has_success_ts).mean()),
    },
    {
        "check": "first_success_before_request",
        "n": int((has_success_ts & (dq["first_success_dttm"] < dq["request_timestamp"])).sum()),
        "share": float(
            (has_success_ts & (dq["first_success_dttm"] < dq["request_timestamp"])).mean()
        ),
    },
    {
        "check": "zero_duration_hours",
        "n": int((has_success_ts & delay_hours_raw.eq(0)).sum()),
        "share": float((has_success_ts & delay_hours_raw.eq(0)).mean()),
    },
    {
        "check": "negative_duration",
        "n": int((has_success_ts & (delay_hours_raw < 0)).sum()),
        "share": float((has_success_ts & (delay_hours_raw < 0)).mean()),
    },
    {
        "check": "extreme_duration_gt_60d",
        "n": int((has_success_ts & (delay_hours_raw / 24.0 > 60)).sum()),
        "share": float((has_success_ts & (delay_hours_raw / 24.0 > 60)).mean()),
    },
    {
        "check": "request_outside_study_window",
        "n": int(
            (
                (dq["request_timestamp"] < STUDY_START)
                | (dq["request_timestamp"] > STUDY_END)
            ).sum()
        ),
        "share": float(
            (
                (dq["request_timestamp"] < STUDY_START)
                | (dq["request_timestamp"] > STUDY_END)
            ).mean()
        ),
    },
    {
        "check": "first_success_after_observation_end",
        "n": int((has_success_ts & (dq["first_success_dttm"] > SUCCESS_OBSERVATION_END)).sum()),
        "share": float(
            (has_success_ts & (dq["first_success_dttm"] > SUCCESS_OBSERVATION_END)).mean()
        ),
    },
]
quality_table = pd.DataFrame(quality_rows)
display(quality_table)
quality_table.to_csv(OUT_DIR / "success_timing_data_quality.csv", index=False)

# Explicit filter rules (reported, then applied):
# 1) Keep applications inside study window (already enforced by build_analysis_panel).
# 2) For delay metrics among successes: require success_flg==1, non-null first_success_dttm,
#    and first_success_dttm >= request_timestamp.
# 3) Extreme delays (>60d) are retained in quality report but flagged; for conditional
#    TTS descriptive plots we keep them unless negative / missing chronology.
# 4) Horizon outcomes already set immature rows to NA via last_mile.outcomes.

VALID_SUCCESS_CHRONOLOGY = (
    success1
    & has_success_ts
    & (dq["first_success_dttm"] >= dq["request_timestamp"])
)
INVALID_SUCCESS_FOR_DELAY = success1 & ~VALID_SUCCESS_CHRONOLOGY

print(
    "[FILTER RULE] valid success chronology for delay metrics: "
    f"{int(VALID_SUCCESS_CHRONOLOGY.sum()):,} "
    f"(excluded success rows: {int(INVALID_SUCCESS_FOR_DELAY.sum()):,})"
)

dq["success_delay_hours"] = np.where(VALID_SUCCESS_CHRONOLOGY, delay_hours_raw, np.nan)
dq["success_delay_days"] = dq["success_delay_hours"] / 24.0
apps_core = dq

print("success_delay_days non-null in apps_core:", int(apps_core["success_delay_days"].notna().sum()))


,check,n,share
0,n_applications,515984,1.000000
1,success_flg_eq_1,345875,0.670321
2,success_flg_eq_0,170109,0.329679
3,success_flg_missing,0,0.000000
4,success1_missing_first_success_dttm,0,0.000000
5,success0_has_first_success_dttm,180,0.000349
6,first_success_before_request,116,0.000225
7,zero_duration_hours,3847,0.007456
8,negative_duration,116,0.000225
9,extreme_duration_gt_60d,28988,0.056180


[FILTER RULE] valid success chronology for delay metrics: 345,759 (excluded success rows: 116)
success_delay_days non-null in apps_core: 345759


## 3. Conditional time-to-success diagnostic (descriptive only)

Только валидные `success_flg == 1` с корректной хронологией.
Это **не** causal estimand.


In [6]:
# ============================================================
# 5. Conditional TTS distributions
# ============================================================

def delay_summary(s: pd.Series, label: str) -> dict:
    x = s.dropna().astype(float)
    if x.empty:
        qs = {f"p{p}": np.nan for p in (10, 25, 50, 75, 90, 95, 99)}
        return {"slice": label, "count": 0, "mean": np.nan, "std": np.nan, **qs}
    q = x.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    return {
        "slice": label,
        "count": int(len(x)),
        "mean": float(x.mean()),
        "std": float(x.std(ddof=1)) if len(x) > 1 else np.nan,
        "p10": float(q.loc[0.10]),
        "p25": float(q.loc[0.25]),
        "median": float(q.loc[0.50]),
        "p75": float(q.loc[0.75]),
        "p90": float(q.loc[0.90]),
        "p95": float(q.loc[0.95]),
        "p99": float(q.loc[0.99]),
    }


succ = apps_core.loc[apps_core["success_delay_days"].notna()].copy()
assert (succ["success_flg"] == 1).all()

delay_rows = [delay_summary(succ["success_delay_days"], "overall")]

# B. pre vs post among treated CORE
treated_succ = succ.loc[succ["change_type"].isin(TYPE_ORDER)].copy()
for period in ("pre", "post"):
    delay_rows.append(
        delay_summary(
            treated_succ.loc[treated_succ["period"] == period, "success_delay_days"],
            f"treated_{period}",
        )
    )

# C. by change_type
for ctype in TYPE_ORDER:
    delay_rows.append(
        delay_summary(
            succ.loc[succ["change_type"] == ctype, "success_delay_days"],
            f"change_type={ctype}",
        )
    )

# D. by treatment cohort
for cohort in sorted(VALID_COHORTS_CONVERSION):
    delay_rows.append(
        delay_summary(
            succ.loc[succ["treatment_date"] == cohort, "success_delay_days"],
            f"cohort={cohort.date()}",
        )
    )

delay_summary_table = pd.DataFrame(delay_rows)
display(delay_summary_table)
delay_summary_table.to_csv(OUT_DIR / "success_timing_summary.csv", index=False)


def add_ref_lines(ax, days=REF_DAYS):
    for d in days:
        ax.axvline(d, color=PALETTE["grid"], linewidth=0.9, linestyle=":", zorder=0)


@with_plot_style
def plot_delay_histograms():
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))

    ax = axes[0]
    ax.hist(
        succ["success_delay_days"],
        bins=40,
        color=PALETTE["treated"],
        edgecolor="white",
        linewidth=0.4,
    )
    add_ref_lines(ax)
    ax.set_xlabel("Days to successful meeting")
    ax.set_ylabel("Count")
    ax.set_title("Histogram of success delay (all valid successes)")
    style_axes(ax)

    ax = axes[1]
    clipped = succ.loc[succ["success_delay_days"].between(0, 25), "success_delay_days"]
    ax.hist(
        clipped,
        bins=np.arange(0, 26.5, 1.0),
        color=PALETTE["treated"],
        edgecolor="white",
        linewidth=0.4,
        histtype="stepfilled",
        alpha=0.85,
    )
    add_ref_lines(ax)
    ax.set_xlim(0, 25)
    ax.set_xlabel("Days to successful meeting (0–25)")
    ax.set_ylabel("Count")
    ax.set_title("Step histogram, 0–25 day window")
    style_axes(ax)

    fig.tight_layout()
    save_figure(fig, FIG_DIR / "hist_success_delay_days.pdf")


@with_plot_style
def plot_ecdf_pre_post():
    fig, ax = plt.subplots(figsize=(6.2, 4.0))
    for period, color, ls in [
        ("pre", PALETTE["control"], "--"),
        ("post", PALETTE["treated"], "-"),
    ]:
        x = np.sort(
            treated_succ.loc[treated_succ["period"] == period, "success_delay_days"]
            .dropna()
            .to_numpy()
        )
        if len(x) == 0:
            continue
        y = np.arange(1, len(x) + 1) / len(x)
        ax.step(x, y, where="post", color=color, linestyle=ls, label=f"{period} (n={len(x)})")
    add_ref_lines(ax)
    ax.set_xlim(left=0)
    ax.set_xlabel("Days to successful meeting")
    ax.set_ylabel("ECDF")
    ax.set_title("ECDF of success delay: pre vs post (treated CORE)")
    ax.legend(loc="lower right")
    style_axes(ax)
    fig.tight_layout()
    save_figure(fig, FIG_DIR / "ecdf_success_delay_pre_post.pdf")


@with_plot_style
def plot_ecdf_by_change_type():
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    for ctype in TYPE_ORDER:
        x = np.sort(
            succ.loc[succ["change_type"] == ctype, "success_delay_days"].dropna().to_numpy()
        )
        if len(x) == 0:
            continue
        y = np.arange(1, len(x) + 1) / len(x)
        ax.step(
            x,
            y,
            where="post",
            color=TYPE_COLORS[ctype],
            label=f"{TYPE_LABELS_EN[ctype]} (n={len(x)})",
        )
    add_ref_lines(ax)
    ax.set_xlim(left=0)
    ax.set_xlabel("Days to successful meeting")
    ax.set_ylabel("ECDF")
    ax.set_title("ECDF of success delay by CORE change type")
    ax.legend(loc="lower right")
    style_axes(ax)
    fig.tight_layout()
    save_figure(fig, FIG_DIR / "ecdf_success_delay_by_change_type.pdf")


plot_delay_histograms()
plot_ecdf_pre_post()
plot_ecdf_by_change_type()
print("Saved conditional TTS figures.")


,slice,count,mean,std,p10,p25,median,p75,p90,p95,p99
0,overall,345759,12.577602,27.638264,2.0,2.0,3.0,6.0,20.0,101.0,110.0
1,treated_pre,99332,15.246185,30.525629,2.0,3.0,4.0,7.0,80.0,104.0,111.0
2,treated_post,26518,4.092956,3.623720,2.0,2.0,3.0,5.0,7.0,10.0,19.0
3,change_type=region_and_workmode,46899,14.499392,29.170035,2.0,3.0,4.0,7.0,41.0,103.0,111.0
4,change_type=region_only,50274,13.016370,28.091349,1.0,2.0,4.0,6.0,23.0,101.0,110.0
5,change_type=workmode_only,93018,12.370659,26.583909,2.0,3.0,4.0,6.0,17.0,100.0,111.0
6,cohort=2022-08-12,9850,14.533096,29.170990,2.0,3.0,4.0,7.0,36.0,102.0,112.0
7,cohort=2022-08-19,59032,11.725166,25.550799,2.0,3.0,4.0,6.0,15.0,99.0,111.0
8,cohort=2022-09-22,56968,13.826359,29.158460,2.0,2.0,4.0,6.0,39.0,102.0,110.0


Saved conditional TTS figures.


## 4. Основной анализ: cumulative success among mature applications

Для каждого горизонта \(H\):

`success_within_H = 1`, если `first_success_dttm` существует и
\(0 \le first\_success\_dttm - request\_timestamp \le H\) дней;
иначе `0`.

Заявка **включается** в анализ горизонта \(H\) только если
`request_timestamp + H <= observation_end` (зрелость).
Незрелые заявки **не** считаются failures (`NA` в outcome).


In [7]:
# ============================================================
# 6. Horizon summary tables (eligible / successes / rate)
# ============================================================

def horizon_slice_summary(df: pd.DataFrame, horizons=HORIZONS_H, slice_name="all") -> pd.DataFrame:
    rows = []
    for H in horizons:
        mcol = f"mature_success_{H}"
        ocol = f"success_within_{H}"
        eligible = df.loc[df[mcol]].copy()
        n_eligible = int(len(eligible))
        n_success = int(eligible[ocol].fillna(0).sum()) if n_eligible else 0
        rate = (n_success / n_eligible) if n_eligible else np.nan
        rows.append(
            {
                "slice": slice_name,
                "horizon_days": H,
                "n_eligible": n_eligible,
                "n_successes": n_success,
                "success_rate": rate,
            }
        )
    return pd.DataFrame(rows)


horizon_parts = [horizon_slice_summary(apps_core, slice_name="ALL_CORE_plus_control")]
# treated pre/post
treated_core = apps_core.loc[apps_core["change_type"].isin(TYPE_ORDER)]
horizon_parts.append(
    horizon_slice_summary(
        treated_core.loc[treated_core["period"] == "pre"], slice_name="treated_pre"
    )
)
horizon_parts.append(
    horizon_slice_summary(
        treated_core.loc[treated_core["period"] == "post"], slice_name="treated_post"
    )
)
horizon_parts.append(
    horizon_slice_summary(apps_core.loc[~apps_core["is_treated"]], slice_name="control")
)

for ctype in TYPE_ORDER:
    horizon_parts.append(
        horizon_slice_summary(
            apps_core.loc[apps_core["change_type"] == ctype],
            slice_name=f"change_type={ctype}",
        )
    )
for cohort in sorted(VALID_COHORTS_CONVERSION):
    horizon_parts.append(
        horizon_slice_summary(
            apps_core.loc[apps_core["treatment_date"] == cohort],
            slice_name=f"cohort={cohort.date()}",
        )
    )

success_horizon_summary = pd.concat(horizon_parts, ignore_index=True)
horizon_preview = success_horizon_summary.loc[
    success_horizon_summary["slice"].isin(
        ["ALL_CORE_plus_control", "treated_pre", "treated_post", "control"]
    )
    & success_horizon_summary["horizon_days"].isin([3, 7, 14, 20, 25])
].copy()
print(
    f"success_horizon_summary rows={len(success_horizon_summary)}; "
    "compact preview: main slices at H∈{3,7,14,20,25}"
)
display(horizon_preview)
success_horizon_summary.to_csv(OUT_DIR / "success_horizon_summary.csv", index=False)


success_horizon_summary rows=72; compact preview: main slices at H∈{3,7,14,20,25}


                    slice  horizon_days  n_eligible  n_successes  success_rate
0   ALL_CORE_plus_control             3      508557       177270      0.348574
1   ALL_CORE_plus_control             7      492143       275553      0.559903
2   ALL_CORE_plus_control            14      468054       282079      0.602663
3   ALL_CORE_plus_control            20      455024       277772      0.610456
4   ALL_CORE_plus_control            25      444611       272621      0.613168
5             treated_pre             3      148146        43737      0.295229
6             treated_pre             7      148146        78012      0.526589
7             treated_pre            14      148146        85426      0.576634
8             treated_pre            20      148146        86984      0.587150
9             treated_pre            25      148146        87587      0.591220
10           treated_post             3       41751        15375      0.368255
11           treated_post             7       35941 

## 5. Cumulative success curves

Для \(H = 0,\ldots,25\) оцениваем \(P(\text{success within } H)\) среди заявок,
зрелых для соответствующего \(H\). Difference curve: post − pre в п.п.
Bootstrap 95% CI с `N_BOOTSTRAP` (по умолчанию 200).


In [8]:
# ============================================================
# 7. Cumulative success curves + bootstrap difference
# ============================================================

def cumulative_success_curve(df: pd.DataFrame, days=CURVE_DAYS) -> pd.DataFrame:
    # Vectorized P(success within H) with maturity at each H.
    # Matches last_mile.outcomes: mature iff request+H <= observation_end;
    # success iff first_success in [request, request+H]. H=0 uses a 0-day window.
    request = pd.to_datetime(df["request_timestamp"], errors="coerce")
    success = pd.to_datetime(df["first_success_dttm"], errors="coerce")
    delay_days = (success - request).dt.total_seconds() / 86400.0
    valid_success = success.notna() & request.notna() & delay_days.ge(0)

    rows = []
    for H in days:
        horizon = pd.Timedelta(days=int(H))
        mature = request.notna() & (request + horizon <= SUCCESS_OBSERVATION_END)
        outcome = valid_success & (success <= request + horizon)
        n_eligible = int(mature.sum())
        n_success = int((mature & outcome).sum())
        rate = n_success / n_eligible if n_eligible else np.nan
        rows.append(
            {
                "horizon_days": H,
                "n_eligible": n_eligible,
                "n_successes": n_success,
                "cum_success_rate": rate,
            }
        )
    return pd.DataFrame(rows)


def bootstrap_curve_diff(
    pre_df: pd.DataFrame,
    post_df: pd.DataFrame,
    days=CURVE_DAYS,
    n_boot: int = N_BOOTSTRAP,
) -> pd.DataFrame:
    base_pre = cumulative_success_curve(pre_df, days)
    base_post = cumulative_success_curve(post_df, days)
    out = base_pre[["horizon_days", "n_eligible"]].rename(
        columns={"n_eligible": "n_eligible_pre"}
    )
    out["n_eligible_post"] = base_post["n_eligible"].to_numpy()
    out["pre_rate"] = base_pre["cum_success_rate"].to_numpy()
    out["post_rate"] = base_post["cum_success_rate"].to_numpy()
    out["diff_pp"] = 100.0 * (out["post_rate"] - out["pre_rate"])

    if n_boot <= 0 or pre_df.empty or post_df.empty:
        out["diff_ci_low"] = np.nan
        out["diff_ci_high"] = np.nan
        return out

    boot = np.empty((n_boot, len(days)), dtype=float)
    pre_idx = np.arange(len(pre_df))
    post_idx = np.arange(len(post_df))
    for b in range(n_boot):
        pre_s = pre_df.iloc[RNG.choice(pre_idx, size=len(pre_idx), replace=True)]
        post_s = post_df.iloc[RNG.choice(post_idx, size=len(post_idx), replace=True)]
        diff = (
            cumulative_success_curve(post_s, days)["cum_success_rate"].to_numpy()
            - cumulative_success_curve(pre_s, days)["cum_success_rate"].to_numpy()
        )
        boot[b, :] = 100.0 * diff
    out["diff_ci_low"] = np.nanpercentile(boot, 2.5, axis=0)
    out["diff_ci_high"] = np.nanpercentile(boot, 97.5, axis=0)
    return out


curve_overall = cumulative_success_curve(apps_core)
curve_pre = cumulative_success_curve(treated_core.loc[treated_core["period"] == "pre"])
curve_post = cumulative_success_curve(treated_core.loc[treated_core["period"] == "post"])
curve_by_type = {
    ctype: cumulative_success_curve(apps_core.loc[apps_core["change_type"] == ctype])
    for ctype in TYPE_ORDER
}

# Sanity: vectorized H in library-enriched columns
for H in [1, 3, 7, 14, 25]:
    mcol, ocol = f"mature_success_{H}", f"success_within_{H}"
    elig = apps_core.loc[apps_core[mcol]]
    lib_rate = float(elig[ocol].fillna(0).mean()) if len(elig) else np.nan
    vec_rate = float(
        curve_overall.loc[curve_overall["horizon_days"] == H, "cum_success_rate"].iloc[0]
    )
    assert np.isclose(lib_rate, vec_rate, equal_nan=True), (H, lib_rate, vec_rate)

diff_pre_post = bootstrap_curve_diff(
    treated_core.loc[treated_core["period"] == "pre"],
    treated_core.loc[treated_core["period"] == "post"],
    n_boot=N_BOOTSTRAP,
)
display(diff_pre_post.head(10))


@with_plot_style
def plot_cum_curves():
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))

    ax = axes[0]
    ax.plot(
        curve_overall["horizon_days"],
        100 * curve_overall["cum_success_rate"],
        color=PALETTE["zero"],
        label="Overall",
        linewidth=1.8,
    )
    ax.plot(
        curve_pre["horizon_days"],
        100 * curve_pre["cum_success_rate"],
        color=PALETTE["control"],
        linestyle="--",
        label="Treated pre",
    )
    ax.plot(
        curve_post["horizon_days"],
        100 * curve_post["cum_success_rate"],
        color=PALETTE["treated"],
        label="Treated post",
    )
    for ctype in TYPE_ORDER:
        c = curve_by_type[ctype]
        if c["n_eligible"].max() < 50:
            continue
        ax.plot(
            c["horizon_days"],
            100 * c["cum_success_rate"],
            color=TYPE_COLORS[ctype],
            alpha=0.85,
            label=TYPE_LABELS_EN[ctype],
        )
    add_ref_lines(ax)
    ax.set_xlabel("Days since application")
    ax.set_ylabel("Cumulative success probability (%)")
    ax.set_title("Cumulative success curves")
    ax.legend(loc="lower right", fontsize=8)
    style_axes(ax)

    ax = axes[1]
    ax.plot(diff_pre_post["horizon_days"], diff_pre_post["diff_pp"], color=PALETTE["treated"], lw=1.8)
    if diff_pre_post["diff_ci_low"].notna().any():
        ax.fill_between(
            diff_pre_post["horizon_days"],
            diff_pre_post["diff_ci_low"],
            diff_pre_post["diff_ci_high"],
            color=PALETTE["treated"],
            alpha=0.18,
            label=f"95% bootstrap CI (B={N_BOOTSTRAP})",
        )
    ax.axhline(0, color=PALETTE["zero"], lw=0.9)
    add_ref_lines(ax)
    ax.set_xlabel("Days since application")
    ax.set_ylabel("Post − pre cumulative success (pp)")
    ax.set_title("Difference curve (treated CORE)")
    ax.legend(loc="best", fontsize=8)
    style_axes(ax)

    fig.tight_layout()
    save_figure(fig, FIG_DIR / "cumulative_success_curves.pdf")


plot_cum_curves()
diff_pre_post.to_csv(OUT_DIR / "success_cumcurve_pre_post_diff.csv", index=False)
print("Saved cumulative success figures.")


,horizon_days,n_eligible_pre,n_eligible_post,pre_rate,post_rate,diff_pp,diff_ci_low,diff_ci_high
0,0,148146,44713,0.001769,0.010444,0.867586,0.768500,0.958586
1,1,148146,44259,0.026298,0.047222,2.092365,1.876967,2.321451
2,2,148146,43096,0.133551,0.215844,8.229301,7.822824,8.634591
3,3,148146,41751,0.295229,0.368255,7.302562,6.793658,7.746712
4,4,148146,40360,0.397480,0.444202,4.672267,4.108473,5.174530
5,5,148146,38916,0.457231,0.498278,4.104696,3.438988,4.665504
6,6,148146,37492,0.497718,0.541582,4.386374,3.810438,4.964438
7,7,148146,35941,0.526589,0.574052,4.746328,4.055147,5.338589
8,8,148146,34468,0.541972,0.592927,5.095466,4.476141,5.704341
9,9,148146,33015,0.552435,0.604786,5.235094,4.654884,5.805654


Saved cumulative success figures.


## 6. Event-time diagnostic

`event_day = application_date − treatment_date`,
`event_week = floor(event_day / 7)`, окно \([-8,+8]\) как в основном event-study.


In [9]:
# ============================================================
# 8. Event-week diagnostics for treated CORE
# ============================================================

ev = treated_core.copy()
ev["application_date"] = ev["request_timestamp"].dt.normalize()
ev["event_day"] = (ev["application_date"] - ev["treatment_date"]).dt.days
ev["event_week"] = np.floor(ev["event_day"] / 7.0).astype("Int64")
ev = ev.loc[
    ev["event_week"].between(EVENT_WEEK_MIN, EVENT_WEEK_MAX)
].copy()

weeks = list(range(EVENT_WEEK_MIN, EVENT_WEEK_MAX + 1))


def event_week_metrics(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for w in weeks:
        g = df.loc[df["event_week"] == w]
        delays = g["success_delay_days"].dropna()
        row = {
            "event_week": w,
            "n_applications": int(len(g)),
            "n_successful": int(delays.shape[0]),
            "median_days_to_success": float(delays.median()) if len(delays) else np.nan,
            "p25_days_to_success": float(delays.quantile(0.25)) if len(delays) else np.nan,
            "p75_days_to_success": float(delays.quantile(0.75)) if len(delays) else np.nan,
        }
        for H in (7, 14, 25):
            mcol = f"mature_success_{H}"
            ocol = f"success_within_{H}"
            elig = g.loc[g[mcol]]
            row[f"n_eligible_{H}"] = int(len(elig))
            row[f"cum_success_{H}"] = (
                float(elig[ocol].fillna(0).mean()) if len(elig) else np.nan
            )
        rows.append(row)
    return pd.DataFrame(rows)


event_week_table = event_week_metrics(ev)
display(event_week_table)
event_week_table.to_csv(OUT_DIR / "success_event_week_descriptives.csv", index=False)


@with_plot_style
def plot_event_week_diagnostics():
    fig, axes = plt.subplots(2, 2, figsize=(11.0, 7.2))

    ax = axes[0, 0]
    ax.plot(
        event_week_table["event_week"],
        event_week_table["median_days_to_success"],
        color=PALETTE["treated"],
        marker="o",
        ms=4,
        label="Median",
    )
    ax.fill_between(
        event_week_table["event_week"],
        event_week_table["p25_days_to_success"],
        event_week_table["p75_days_to_success"],
        color=PALETTE["treated"],
        alpha=0.18,
        label="p25–p75",
    )
    ax.axvline(-0.5, color=PALETTE["grid"], ls=":", lw=1)
    ax.set_title("Median time-to-success by event week")
    ax.set_xlabel("Event week")
    ax.set_ylabel("Days to success (conditional)")
    ax.legend(fontsize=8)
    style_axes(ax)

    ax = axes[0, 1]
    ax.bar(
        event_week_table["event_week"],
        event_week_table["n_successful"],
        color=PALETTE["treated"],
        width=0.8,
    )
    for _, r in event_week_table.iterrows():
        ax.text(
            r["event_week"],
            r["n_successful"],
            str(int(r["n_applications"])),
            ha="center",
            va="bottom",
            fontsize=7,
            color=PALETTE["text_muted"],
        )
    ax.set_title("Successful applications by event week")
    ax.set_xlabel("Event week (labels = n applications)")
    ax.set_ylabel("N successful")
    style_axes(ax)

    ax = axes[1, 0]
    for H, color in [(7, PALETTE["region_only"]), (14, PALETTE["workmode_only"]), (25, PALETTE["region_and_workmode"])]:
        ax.plot(
            event_week_table["event_week"],
            100 * event_week_table[f"cum_success_{H}"],
            marker="o",
            ms=3.5,
            color=color,
            label=f"Within {H}d",
        )
    ax.axvline(-0.5, color=PALETTE["grid"], ls=":", lw=1)
    ax.set_title("Cumulative success by event week")
    ax.set_xlabel("Event week")
    ax.set_ylabel("Success rate among eligible (%)")
    ax.legend(fontsize=8)
    style_axes(ax)

    ax = axes[1, 1]
    ax.bar(
        event_week_table["event_week"] - 0.2,
        event_week_table["n_eligible_7"],
        width=0.2,
        color=PALETTE["region_only"],
        label="Eligible H=7",
    )
    ax.bar(
        event_week_table["event_week"],
        event_week_table["n_eligible_14"],
        width=0.2,
        color=PALETTE["workmode_only"],
        label="Eligible H=14",
    )
    ax.bar(
        event_week_table["event_week"] + 0.2,
        event_week_table["n_eligible_25"],
        width=0.2,
        color=PALETTE["region_and_workmode"],
        label="Eligible H=25",
    )
    ax.set_title("Sample size (eligible) by event week")
    ax.set_xlabel("Event week")
    ax.set_ylabel("N eligible")
    ax.legend(fontsize=8)
    style_axes(ax)

    fig.tight_layout()
    save_figure(fig, FIG_DIR / "event_week_success_diagnostics.pdf")


plot_event_week_diagnostics()
print("Saved event-week diagnostic figures.")


,event_week,n_applications,n_successful,median_days_to_success,p25_days_to_success,p75_days_to_success,n_eligible_7,cum_success_7,n_eligible_14,cum_success_14,n_eligible_25,cum_success_25
0,-8,8417,5863,6.0,3.0,103.0,8417,0.375787,8417,0.425567,8417,0.443626
1,-7,6329,4223,4.0,3.0,10.0,6329,0.468794,6329,0.522673,6329,0.538158
2,-6,5768,3844,3.0,3.0,6.0,5768,0.553051,5768,0.608010,5768,0.622920
3,-5,5877,3921,3.0,2.0,6.0,5877,0.567807,5877,0.617832,5877,0.637400
4,-4,5676,3849,4.0,3.0,6.0,5676,0.560958,5676,0.631078,5676,0.653453
5,-3,5813,3921,3.0,2.0,5.0,5813,0.596938,5813,0.648890,5813,0.664029
6,-2,5886,4106,3.0,2.0,5.0,5886,0.623004,5886,0.671254,5886,0.687054
7,-1,5714,3687,3.0,2.0,5.0,5714,0.571229,5714,0.616381,5714,0.634582
8,0,5549,3609,3.0,2.0,6.0,5549,0.557037,5549,0.630023,3723,0.650282
9,1,5711,3718,3.0,2.0,5.0,5711,0.579058,5306,0.640407,3204,0.666042


Saved event-week diagnostic figures.


## 7. Heatmap: event week × success-delay bucket

Значение ячейки — доля **успешных** заявок в соответствующем delay bucket
среди успешных заявок в данном `event_week`
(denominator = successful applications with valid delay in that week).


In [10]:
# ============================================================
# 9. Heatmaps (share within successful applications)
# ============================================================

DELAY_BUCKETS = [
    (0, 1, "0–1"),
    (1, 3, "1–3"),
    (3, 5, "3–5"),
    (5, 7, "5–7"),
    (7, 10, "7–10"),
    (10, 14, "10–14"),
    (14, 20, "14–20"),
    (20, 25, "20–25"),
]
BUCKET_LABELS = [b[2] for b in DELAY_BUCKETS]


def delay_bucket(days: float) -> str | None:
    if pd.isna(days):
        return None
    for lo, hi, lab in DELAY_BUCKETS:
        if lo <= days < hi or (hi == 25 and lo <= days <= hi):
            return lab
    return None


def make_heatmap_matrix(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    s = df.loc[df["success_delay_days"].notna()].copy()
    s = s.loc[s["event_week"].between(EVENT_WEEK_MIN, EVENT_WEEK_MAX)]
    s["bucket"] = s["success_delay_days"].map(delay_bucket)
    s = s.loc[s["bucket"].notna()]
    denom = s.groupby("event_week").size().reindex(weeks, fill_value=0)
    mat = (
        s.groupby(["event_week", "bucket"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=weeks, columns=BUCKET_LABELS, fill_value=0)
    )
    share = mat.div(denom.replace(0, np.nan), axis=0)
    return share, denom


@with_plot_style
def plot_heatmaps():
    panels = [("ALL CORE", ev)]
    for ctype in TYPE_ORDER:
        sub = ev.loc[ev["change_type"] == ctype]
        if sub["success_delay_days"].notna().sum() < 30:
            print(f"[SKIP heatmap] {ctype}: insufficient successful apps")
            continue
        panels.append((TYPE_LABELS_EN[ctype], sub))

    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 4.6), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, (title, sub) in zip(axes, panels):
        share, denom = make_heatmap_matrix(sub)
        im = ax.imshow(
            share.to_numpy(dtype=float),
            aspect="auto",
            cmap="Blues",
            vmin=0,
            vmax=np.nanmax(share.to_numpy()) if np.isfinite(np.nanmax(share.to_numpy())) else 1,
            origin="lower",
        )
        ax.set_xticks(range(len(BUCKET_LABELS)))
        ax.set_xticklabels(BUCKET_LABELS, rotation=45, ha="right")
        ax.set_yticks(range(len(weeks)))
        ax.set_yticklabels([str(w) for w in weeks])
        ax.set_title(f"{title}\n(denom = successful apps)")
        ax.set_xlabel("Success delay bucket (days)")
        ax.set_ylabel("Event week")
        # Annotate thin n on right via secondary text under title
        ax.text(
            0.0,
            1.02,
            f"min n={int(denom.min())}, max n={int(denom.max())}",
            transform=ax.transAxes,
            fontsize=7,
            color=PALETTE["text_muted"],
        )

    fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label="Share of successes")
    fig.tight_layout()
    save_figure(fig, FIG_DIR / "heatmap_event_week_delay_buckets.pdf")


# event_week already on ev
plot_heatmaps()
print("Saved heatmaps.")


Saved heatmaps.


## 8. Multi-horizon DiD / event-study

Та же FE-спецификация, что в `final_empirical_recalculation.ipynb`:

- unit: hex × calendar day (binary outcome → mean rate)
- hex FE + calendar-day FE (`PanelOLS`)
- clustered SE by hex
- event weeks \([-8,+8]\), baseline week = `-1`
- pretrend: joint Wald на неделях \(< -1\)
- pooled post: weighted average of post coefficients by `n_treated_orders`

Исходы: `success_within_{3,7,14,25}` по каждому CORE `change_type`.


In [11]:
# ============================================================
# 10. Estimator helpers (aligned with final_empirical_recalculation)
# ============================================================

class EmptyWaldResult:
    pval = np.nan
    stat = np.nan


def maturity_horizon_days(outcome: str) -> int:
    if outcome.startswith("success_within_"):
        return int(outcome.rsplit("_", 1)[1])
    return 0


def filter_mature_requests(df: pd.DataFrame, outcome: str):
    h = maturity_horizon_days(outcome)
    if h <= 0:
        return df.copy(), h
    obs_end = SUCCESS_OBSERVATION_END
    return df[df["request_timestamp"] + pd.Timedelta(days=h) <= obs_end].copy(), h


def make_treated_sample(outcome: str, change_type: str) -> pd.DataFrame:
    cohorts = cohorts_for_outcome(outcome)
    t = apps_t[
        (apps_t["change_type"] == change_type)
        & (apps_t["treatment_date"].isin(cohorts))
    ].copy()
    t, _ = filter_mature_requests(t, outcome)
    return t


def make_control_sample(outcome: str) -> pd.DataFrame:
    c, _ = filter_mature_requests(apps_c.copy(), outcome)
    return c


def week_name(w: int) -> str:
    return f"pre{-w}" if w < 0 else f"post{w}"


def _full_rank_event_columns(d: pd.DataFrame, reg_cols: list[str]) -> list[str]:
    active = []
    for col in reg_cols:
        if col not in d.columns:
            continue
        if d[col].sum() == 0 or d[col].nunique(dropna=False) < 2:
            continue
        active.append(col)
    kept = []
    for col in active:
        candidate = kept + [col]
        x = d[candidate].to_numpy(dtype=float)
        if np.linalg.matrix_rank(x) > len(kept):
            kept.append(col)
    return kept


def did_event_study(outcome: str, change_type: str, max_week: int = 8):
    name = week_name
    weeks_es = [w for w in range(-max_week, max_week + 1) if w != -1]

    t = make_treated_sample(outcome, change_type)
    if t.empty:
        raise ValueError(f"Empty treated sample: {outcome} × {change_type}")

    t["rel_week"] = (t["days_from_treatment"] // 7).clip(-max_week, max_week)
    t["is_treated_es"] = 1

    c = make_control_sample(outcome)
    c["rel_week"] = np.nan
    c["is_treated_es"] = 0

    d = pd.concat([t, c], ignore_index=True)
    d["date"] = d["request_timestamp"].dt.normalize()
    for w in weeks_es:
        d[name(w)] = ((d["is_treated_es"] == 1) & (d["rel_week"] == w)).astype(int)

    all_reg_cols = [name(w) for w in weeks_es]
    d = d.dropna(subset=[outcome]).copy()

    treated_support_base = d[(d["is_treated_es"] == 1) & d["rel_week"].notna()].copy()
    treated_support_base["rel_week"] = treated_support_base["rel_week"].astype(int)
    n_orders = treated_support_base.groupby("rel_week")["request_timestamp"].count()
    n_hex = treated_support_base.groupby("rel_week")["hex"].nunique()
    n_hex_days = (
        treated_support_base[["rel_week", "hex", "date"]]
        .drop_duplicates()
        .groupby("rel_week")
        .size()
    )
    n_cohorts = treated_support_base.groupby("rel_week")["treatment_date"].nunique()
    event_week_support = (
        pd.DataFrame(
            {
                "n_treated_orders": n_orders,
                "n_treated_hex": n_hex,
                "n_treated_hex_days": n_hex_days,
                "n_supporting_cohorts": n_cohorts,
            }
        )
        .reindex(weeks_es)
        .fillna(0)
        .astype(int)
    )

    agg_dict = {outcome: "mean", "request_timestamp": "count"}
    agg_dict.update({col: "max" for col in all_reg_cols})
    reg_panel = (
        d.groupby(["hex", "date"], as_index=False)
        .agg(agg_dict)
        .rename(columns={"request_timestamp": "n_orders"})
        .set_index(["hex", "date"])
        .sort_index()
    )
    if not reg_panel.index.is_unique:
        raise ValueError(f"Non-unique hex×date for {outcome} × {change_type}")

    reg_cols = _full_rank_event_columns(reg_panel, all_reg_cols)
    if not reg_cols:
        raise ValueError(f"No estimable event-week dummies for {outcome} × {change_type}")

    res = PanelOLS(
        reg_panel[outcome],
        reg_panel[reg_cols],
        entity_effects=True,
        time_effects=True,
        drop_absorbed=True,
    ).fit(cov_type="clustered", cluster_entity=True, low_memory=True)

    ci = res.conf_int()
    rows = []
    for w in weeks_es:
        col = name(w)
        if col in res.params.index:
            rows.append(
                {
                    "rel_week": w,
                    "coef": float(res.params[col]),
                    "ci_low": float(ci.loc[col, "lower"]),
                    "ci_high": float(ci.loc[col, "upper"]),
                    "pval": float(res.pvalues[col]),
                    "estimated": True,
                }
            )
        else:
            rows.append(
                {
                    "rel_week": w,
                    "coef": np.nan,
                    "ci_low": np.nan,
                    "ci_high": np.nan,
                    "pval": np.nan,
                    "estimated": False,
                }
            )
    coefs = pd.DataFrame(rows).set_index("rel_week").sort_index()

    pre_cols = [name(w) for w in weeks_es if w < -1 and name(w) in res.params.index]
    if pre_cols:
        wald = res.wald_test(formula=", ".join(f"{col} = 0" for col in pre_cols))
    else:
        wald = EmptyWaldResult()

    meta = {
        "n_obs_hex_day": int(res.nobs),
        "n_hex": int(reg_panel.index.get_level_values(0).nunique()),
        "n_treated_orders": int(len(t)),
        "n_control_orders": int(len(c)),
    }
    return res, coefs, wald, event_week_support, meta


def summarize_post_effect(coefs, support, res=None, min_supporting_cohorts=None) -> dict:
    weight_col = "n_treated_orders"
    post = coefs.loc[coefs.index >= 0, ["coef"]].copy().join(support, how="left")
    post[weight_col] = post[weight_col].fillna(0)
    if "n_supporting_cohorts" not in post.columns:
        post["n_supporting_cohorts"] = np.nan
    usable = post[post["coef"].notna() & (post[weight_col] > 0)].copy()
    if min_supporting_cohorts is not None:
        usable = usable[usable["n_supporting_cohorts"] >= min_supporting_cohorts].copy()
    if usable.empty:
        return {
            "avg_post_effect_weighted": np.nan,
            "avg_post_se": np.nan,
            "avg_post_ci_low": np.nan,
            "avg_post_ci_high": np.nan,
            "post_weeks": [],
        }
    weights = usable[weight_col].to_numpy(dtype=float)
    a = weights / weights.sum()
    beta = usable["coef"].to_numpy(dtype=float)
    weighted = float(a @ beta)
    post_weeks = [int(w) for w in usable.index.tolist()]
    avg_se = ci_low = ci_high = np.nan
    if res is not None:
        post_cols = [week_name(w) for w in post_weeks]
        if not any(c not in res.params.index for c in post_cols):
            cov = res.cov.loc[post_cols, post_cols].to_numpy(dtype=float)
            avg_se = float(np.sqrt(max(float(a @ cov @ a), 0.0)))
            ci_low = weighted - 1.96 * avg_se
            ci_high = weighted + 1.96 * avg_se
    return {
        "avg_post_effect_weighted": weighted,
        "avg_post_se": avg_se,
        "avg_post_ci_low": ci_low,
        "avg_post_ci_high": ci_high,
        "post_weeks": post_weeks,
    }


print("Estimator helpers ready.")


Estimator helpers ready.


In [12]:
# ============================================================
# 11. Run multi-horizon event studies
# ============================================================

DID_OUTCOMES = [f"success_within_{H}" for H in DID_HORIZONS]

es_coef_rows = []
did_summary_rows = []

for outcome in DID_OUTCOMES:
    H = int(outcome.rsplit("_", 1)[1])
    for ctype in TYPE_ORDER:
        print(f"Estimating {outcome} × {ctype} ...")
        try:
            res, coefs, wald, support, meta = did_event_study(outcome, ctype, max_week=8)
        except Exception as exc:
            print(f"  SKIP: {type(exc).__name__}: {exc}")
            did_summary_rows.append(
                {
                    "change_type": ctype,
                    "horizon_days": H,
                    "outcome": outcome,
                    "N": np.nan,
                    "n_hex": np.nan,
                    "pre_rate": np.nan,
                    "post_rate": np.nan,
                    "raw_diff_pp": np.nan,
                    "DID_post_estimate_pp": np.nan,
                    "SE_pp": np.nan,
                    "CI_low_pp": np.nan,
                    "CI_high_pp": np.nan,
                    "pretrend_pvalue": np.nan,
                    "status": f"error:{type(exc).__name__}",
                }
            )
            continue

        post = summarize_post_effect(coefs, support, res=res)

        # Raw pre/post rates on treated sample (descriptive companion)
        t_sample = make_treated_sample(outcome, ctype)
        t_sample = t_sample.dropna(subset=[outcome])
        pre_rate = float(
            t_sample.loc[t_sample["days_from_treatment"] < 0, outcome].astype(float).mean()
        ) if (t_sample["days_from_treatment"] < 0).any() else np.nan
        post_rate = float(
            t_sample.loc[t_sample["days_from_treatment"] >= 0, outcome].astype(float).mean()
        ) if (t_sample["days_from_treatment"] >= 0).any() else np.nan
        raw_diff_pp = (
            100.0 * (post_rate - pre_rate)
            if np.isfinite(pre_rate) and np.isfinite(post_rate)
            else np.nan
        )

        pretrend_p = float(getattr(wald, "pval", np.nan))
        did_pp = 100.0 * post["avg_post_effect_weighted"] if np.isfinite(post["avg_post_effect_weighted"]) else np.nan
        se_pp = 100.0 * post["avg_post_se"] if np.isfinite(post["avg_post_se"]) else np.nan
        ci_lo = 100.0 * post["avg_post_ci_low"] if np.isfinite(post["avg_post_ci_low"]) else np.nan
        ci_hi = 100.0 * post["avg_post_ci_high"] if np.isfinite(post["avg_post_ci_high"]) else np.nan

        did_summary_rows.append(
            {
                "change_type": ctype,
                "horizon_days": H,
                "outcome": outcome,
                "N": meta["n_obs_hex_day"],
                "n_hex": meta["n_hex"],
                "n_treated_orders": meta["n_treated_orders"],
                "pre_rate": pre_rate,
                "post_rate": post_rate,
                "raw_diff_pp": raw_diff_pp,
                "DID_post_estimate_pp": did_pp,
                "SE_pp": se_pp,
                "CI_low_pp": ci_lo,
                "CI_high_pp": ci_hi,
                "pretrend_pvalue": pretrend_p,
                "post_weeks": ",".join(str(w) for w in post["post_weeks"]),
                "status": "ok",
            }
        )

        coef_reset = coefs.reset_index()
        coef_reset["change_type"] = ctype
        coef_reset["horizon_days"] = H
        coef_reset["outcome"] = outcome
        coef_reset["pretrend_pvalue"] = pretrend_p
        es_coef_rows.append(coef_reset)

success_event_study = pd.concat(es_coef_rows, ignore_index=True) if es_coef_rows else pd.DataFrame()
success_horizon_did = pd.DataFrame(did_summary_rows)

display(success_horizon_did)
success_event_study.to_csv(OUT_DIR / "success_event_study.csv", index=False)
success_horizon_did.to_csv(OUT_DIR / "success_horizon_did.csv", index=False)


Estimating success_within_3 × region_and_workmode ...
Estimating success_within_3 × region_only ...
Estimating success_within_3 × workmode_only ...
Estimating success_within_7 × region_and_workmode ...
Estimating success_within_7 × region_only ...
Estimating success_within_7 × workmode_only ...
Estimating success_within_14 × region_and_workmode ...
Estimating success_within_14 × region_only ...
Estimating success_within_14 × workmode_only ...
Estimating success_within_25 × region_and_workmode ...
Estimating success_within_25 × region_only ...
Estimating success_within_25 × workmode_only ...


,change_type,horizon_days,outcome,N,n_hex,n_treated_orders,pre_rate,post_rate,raw_diff_pp,DID_post_estimate_pp,SE_pp,CI_low_pp,CI_high_pp,pretrend_pvalue,post_weeks,status
0,region_and_workmode,3,success_within_3,100900,6145,55570,0.275983,0.249385,-2.659850,-4.592167,2.085762,-8.680262,-0.504073,0.009874,"0,1,2,3,4,5,6,7,8",ok
1,region_only,3,success_within_3,89525,4456,31920,0.378496,0.407909,2.941388,-2.049003,2.633177,-7.210029,3.112024,0.704148,"0,1,2,3,4,5,6,7,8",ok
2,workmode_only,3,success_within_3,111680,5549,102407,0.277713,0.403788,12.607435,3.292988,1.560209,0.234977,6.350998,0.851251,"0,1,2,3,4,5,6,7,8",ok
3,region_and_workmode,7,success_within_7,98181,6138,54054,0.500502,0.484944,-1.555784,-6.182348,2.375191,-10.837722,-1.526974,0.225716,"0,1,2,3,4,5,6,7,8",ok
4,region_only,7,success_within_7,87121,4451,30892,0.561167,0.586508,2.534077,-4.142540,2.955198,-9.934728,1.649648,0.145174,"0,1,2,3,4,5,6,7,8",ok
5,workmode_only,7,success_within_7,108657,5544,99141,0.530253,0.603161,7.290795,1.735665,1.757052,-1.708157,5.179486,0.001524,"0,1,2,3,4,5,6,7,8",ok
6,region_and_workmode,14,success_within_14,93947,6118,51639,0.556463,0.579454,2.299118,-2.382719,2.351599,-6.991854,2.226416,0.217940,"0,1,2,3,4,5,6,7",ok
7,region_only,14,success_within_14,83352,4435,29433,0.596555,0.630562,3.400690,-3.291396,3.053415,-9.276089,2.693298,0.170231,"0,1,2,3,4,5,6,7",ok
8,workmode_only,14,success_within_14,103845,5530,94073,0.581853,0.654576,7.272284,0.660310,1.747611,-2.765007,4.085628,0.090143,"0,1,2,3,4,5,6,7",ok
9,region_and_workmode,25,success_within_25,89172,6088,49229,0.573596,0.619466,4.587090,-0.504075,2.583387,-5.567515,4.559364,0.167424,"0,1,2,3,4,5,6",ok


In [13]:
# ============================================================
# 12. Comparison plot across horizons
# ============================================================

@with_plot_style
def plot_horizon_did_comparison():
    fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.8), sharey=True)
    for ax, ctype in zip(axes, TYPE_ORDER):
        sub = success_horizon_did.loc[
            (success_horizon_did["change_type"] == ctype)
            & (success_horizon_did["status"] == "ok")
        ].sort_values("horizon_days")
        if sub.empty:
            ax.set_title(TYPE_LABELS_EN[ctype])
            continue
        ax.errorbar(
            sub["horizon_days"],
            sub["DID_post_estimate_pp"],
            yerr=1.96 * sub["SE_pp"],
            fmt="o-",
            color=TYPE_COLORS[ctype],
            ecolor=PALETTE["text_muted"],
            capsize=3,
            lw=1.4,
        )
        ax.axhline(0, color=PALETTE["zero"], lw=0.9)
        for _, r in sub.iterrows():
            mark = ""
            if pd.notna(r["pretrend_pvalue"]) and r["pretrend_pvalue"] < 0.05:
                mark = "*"
            ax.text(
                r["horizon_days"],
                r["DID_post_estimate_pp"],
                mark,
                ha="left",
                va="bottom",
                fontsize=10,
                color=PALETTE["text"],
            )
        ax.set_xlabel("Horizon H (days)")
        ax.set_title(TYPE_LABELS_EN[ctype])
        style_axes(ax)
    axes[0].set_ylabel("Weighted post DiD estimate (pp)")
    fig.suptitle(
        "Multi-horizon event-study post effects (* = pretrend p<0.05)",
        y=1.02,
        fontsize=10,
    )
    fig.tight_layout()
    save_figure(fig, FIG_DIR / "multi_horizon_did_comparison.pdf")


@with_plot_style
def plot_event_study_grid():
    fig, axes = plt.subplots(
        len(DID_HORIZONS),
        len(TYPE_ORDER),
        figsize=(11.5, 2.6 * len(DID_HORIZONS)),
        sharex=True,
        sharey=True,
    )
    for i, H in enumerate(DID_HORIZONS):
        for j, ctype in enumerate(TYPE_ORDER):
            ax = axes[i, j]
            sub = success_event_study.loc[
                (success_event_study["horizon_days"] == H)
                & (success_event_study["change_type"] == ctype)
            ].sort_values("rel_week")
            if sub.empty:
                ax.set_axis_off()
                continue
            ax.axhline(0, color=PALETTE["zero"], lw=0.8)
            ax.axvline(-0.5, color=PALETTE["grid"], ls=":", lw=0.9)
            ax.fill_between(
                sub["rel_week"],
                100 * sub["ci_low"],
                100 * sub["ci_high"],
                color=TYPE_COLORS[ctype],
                alpha=0.15,
            )
            ax.plot(
                sub["rel_week"],
                100 * sub["coef"],
                color=TYPE_COLORS[ctype],
                marker="o",
                ms=3,
            )
            # omitted baseline marker
            ax.scatter([-1], [0], color=PALETTE["zero"], zorder=5, s=18)
            if i == 0:
                ax.set_title(TYPE_LABELS_EN[ctype], fontsize=9)
            if j == 0:
                ax.set_ylabel(f"H={H}\npp", fontsize=8)
            if i == len(DID_HORIZONS) - 1:
                ax.set_xlabel("Event week")
            style_axes(ax)
    fig.suptitle("Event-study coefficients by success horizon", y=1.01, fontsize=10)
    fig.tight_layout()
    save_figure(fig, FIG_DIR / "event_study_by_horizon.pdf")


if not success_horizon_did.empty:
    plot_horizon_did_comparison()
if not success_event_study.empty:
    plot_event_study_grid()
print("Saved DiD comparison figures.")


Saved DiD comparison figures.


## 9–10. Summary table and automatic interpretation

Итоговая таблица объединяет descriptive pre/post rates и weighted post DiD-оценки.
Автоинтерпретация **не** делает causal claims при нарушенных pretrends.


In [14]:
# ============================================================
# 13. Final summary table (alias columns as requested)
# ============================================================

summary_out = success_horizon_did.rename(
    columns={
        "DID_post_estimate_pp": "DID_post_estimate_pp",
        "SE_pp": "SE",
        "CI_low_pp": "CI_low",
        "CI_high_pp": "CI_high",
    }
)[
    [
        "change_type",
        "horizon_days",
        "N",
        "pre_rate",
        "post_rate",
        "raw_diff_pp",
        "DID_post_estimate_pp",
        "SE",
        "CI_low",
        "CI_high",
        "pretrend_pvalue",
        "n_hex",
        "status",
    ]
].copy()

display(summary_out)
summary_out.to_csv(OUT_DIR / "success_timing_did_summary.csv", index=False)
# Keep requested filename alias
summary_out.to_csv(OUT_DIR / "success_horizon_did.csv", index=False)


,change_type,horizon_days,N,pre_rate,post_rate,raw_diff_pp,DID_post_estimate_pp,SE,CI_low,CI_high,pretrend_pvalue,n_hex,status
0,region_and_workmode,3,100900,0.275983,0.249385,-2.659850,-4.592167,2.085762,-8.680262,-0.504073,0.009874,6145,ok
1,region_only,3,89525,0.378496,0.407909,2.941388,-2.049003,2.633177,-7.210029,3.112024,0.704148,4456,ok
2,workmode_only,3,111680,0.277713,0.403788,12.607435,3.292988,1.560209,0.234977,6.350998,0.851251,5549,ok
3,region_and_workmode,7,98181,0.500502,0.484944,-1.555784,-6.182348,2.375191,-10.837722,-1.526974,0.225716,6138,ok
4,region_only,7,87121,0.561167,0.586508,2.534077,-4.142540,2.955198,-9.934728,1.649648,0.145174,4451,ok
5,workmode_only,7,108657,0.530253,0.603161,7.290795,1.735665,1.757052,-1.708157,5.179486,0.001524,5544,ok
6,region_and_workmode,14,93947,0.556463,0.579454,2.299118,-2.382719,2.351599,-6.991854,2.226416,0.217940,6118,ok
7,region_only,14,83352,0.596555,0.630562,3.400690,-3.291396,3.053415,-9.276089,2.693298,0.170231,4435,ok
8,workmode_only,14,103845,0.581853,0.654576,7.272284,0.660310,1.747611,-2.765007,4.085628,0.090143,5530,ok
9,region_and_workmode,25,89172,0.573596,0.619466,4.587090,-0.504075,2.583387,-5.567515,4.559364,0.167424,6088,ok


In [15]:
# ============================================================
# 14. Automatic interpretation
# ============================================================

def classify_case(df_ctype: pd.DataFrame) -> tuple[str, bool]:
    # Heuristic case label from multi-horizon DiD estimates (pp).
    ok = df_ctype.loc[df_ctype["status"] == "ok"].copy()
    if ok.empty:
        return "CASE 4", False

    pretrend_fail = bool((ok["pretrend_pvalue"].fillna(1.0) < 0.05).any())
    half_width = 1.96 * ok["SE"]
    wide = bool(ok["SE"].isna().all()) or bool((half_width.fillna(np.inf) > 8).mean() >= 0.5)

    def est(H):
        rows = ok.loc[ok["horizon_days"] == H, "DID_post_estimate_pp"]
        return float(rows.iloc[0]) if len(rows) else np.nan

    e3, e7, e14, e25 = est(3), est(7), est(14), est(25)
    short = np.nanmean([e3, e7])
    long = e25

    if wide or ok["DID_post_estimate_pp"].isna().all():
        case = "CASE 4"
    elif np.isfinite(short) and np.isfinite(long) and short > 1.0 and abs(long) <= 1.0:
        case = "CASE 1"
    elif all(np.isfinite(x) and abs(x) <= 1.0 for x in [e3, e7, e14, e25] if np.isfinite(x)):
        case = "CASE 2"
    elif np.isfinite(long) and abs(long) > 1.0 and (
        (not np.isfinite(short)) or abs(short) <= 1.0
    ):
        case = "CASE 3"
    else:
        case = "CASE 4"

    return case, pretrend_fail


CASE_TEXT = {
    "CASE 1": (
        "changes appear to accelerate successful meetings without materially changing "
        "the final 25-day success probability."
    ),
    "CASE 2": (
        "there is no detectable evidence that the changes affected either the timing "
        "or the final probability of successful meetings."
    ),
    "CASE 3": (
        "the changes may affect eventual conversion rather than immediate "
        "scheduling/success speed."
    ),
    "CASE 4": (
        "the data do not allow a precise conclusion about changes in the timing of success."
    ),
}

# A. Descriptive evidence
desc_lines = []
ov = success_horizon_summary.loc[
    success_horizon_summary["slice"] == "ALL_CORE_plus_control"
]
for H in [3, 7, 14, 25]:
    r = ov.loc[ov["horizon_days"] == H]
    if not r.empty:
        desc_lines.append(
            f"- Overall mature success_within_{H}: "
            f"rate={100*float(r['success_rate'].iloc[0]):.2f}% "
            f"(N={int(r['n_eligible'].iloc[0]):,})"
        )

pre = success_horizon_summary.loc[success_horizon_summary["slice"] == "treated_pre"]
post = success_horizon_summary.loc[success_horizon_summary["slice"] == "treated_post"]
for H in [3, 7, 25]:
    rp = pre.loc[pre["horizon_days"] == H]
    rq = post.loc[post["horizon_days"] == H]
    if not rp.empty and not rq.empty:
        diff = 100 * (float(rq["success_rate"].iloc[0]) - float(rp["success_rate"].iloc[0]))
        desc_lines.append(
            f"- Treated pre→post raw diff at H={H}: {diff:+.2f} pp"
        )

median_overall = delay_summary_table.loc[
    delay_summary_table["slice"] == "overall", "median"
]
if not median_overall.empty:
    desc_lines.append(
        f"- Conditional median days-to-success (successes only): "
        f"{float(median_overall.iloc[0]):.2f} days "
        f"[DESCRIPTIVE / CONDITIONED-ON-OUTCOME]"
    )

# B. DiD evidence
did_lines = []
case_by_type = {}
for ctype in TYPE_ORDER:
    sub = summary_out.loc[summary_out["change_type"] == ctype]
    case, pretrend_fail = classify_case(sub)
    case_by_type[ctype] = (case, pretrend_fail)
    did_lines.append(f"### {TYPE_LABELS_EN[ctype]} (`{ctype}`)")
    did_lines.append(f"- Heuristic pattern: **{case}** — {CASE_TEXT[case]}")
    if pretrend_fail:
        did_lines.append(
            "- Pretrends: rejected at 5% for at least one horizon → "
            "**no causal claim** for this change type."
        )
    else:
        did_lines.append("- Pretrends: not rejected at 5% for reported horizons (still diagnostic).")
    for _, r in sub.sort_values("horizon_days").iterrows():
        did_lines.append(
            f"- H={int(r['horizon_days'])}: DiD={r['DID_post_estimate_pp']:.2f} pp "
            f"(SE={r['SE']:.2f}, CI=[{r['CI_low']:.2f}, {r['CI_high']:.2f}]), "
            f"pretrend p={r['pretrend_pvalue']:.3f}, N={r['N']}, n_hex={r['n_hex']}"
            if r["status"] == "ok" and pd.notna(r["DID_post_estimate_pp"])
            else f"- H={int(r['horizon_days'])}: estimation unavailable ({r['status']})"
        )

# C. Limitations
lim_lines = [
    "- Conditional TTS among successes is conditioned-on-outcome and not causal.",
    "- Horizon outcomes depend on maturity / right-censoring at observation_end=2022-10-18.",
    "- Composition: CORE only; treated cohort 2022-07-27 excluded for success specs.",
    "- TWFE event-study assumes parallel trends; trust estimates only when pretrends hold.",
    "- Bootstrap CI on descriptive cumulative curves does not replace clustered DiD inference.",
    "- Sample sizes vary by event week and horizon; sparse cells should not be over-interpreted.",
]

md_text = ["## Automatic interpretation summary", "", "### A. Descriptive evidence", *desc_lines, ""]
md_text += ["### B. DiD / event-study evidence", *did_lines, ""]
md_text += ["### C. Limitations", *lim_lines, ""]
md_text += [
    "### Bottom line",
    (
        "Overall heuristic across CORE types: "
        + ", ".join(f"{k}→{v[0]}" for k, v in case_by_type.items())
        + ". Causal language is withheld wherever pretrends fail or precision is weak."
    ),
]

display(Markdown("\n".join(md_text)))

(OUT_DIR / "automatic_interpretation.md").write_text(
    "\n".join(md_text), encoding="utf-8"
)
print(
    "Wrote",
    (OUT_DIR / "automatic_interpretation.md").relative_to(PROJECT_ROOT).as_posix(),
)


## Automatic interpretation summary

### A. Descriptive evidence
- Overall mature success_within_3: rate=34.86% (N=508,557)
- Overall mature success_within_7: rate=55.99% (N=492,143)
- Overall mature success_within_14: rate=60.27% (N=468,054)
- Overall mature success_within_25: rate=61.32% (N=444,611)
- Treated pre→post raw diff at H=3: +7.30 pp
- Treated pre→post raw diff at H=7: +4.75 pp
- Treated pre→post raw diff at H=25: +6.56 pp
- Conditional median days-to-success (successes only): 3.00 days [DESCRIPTIVE / CONDITIONED-ON-OUTCOME]

### B. DiD / event-study evidence
### Region + work mode (`region_and_workmode`)
- Heuristic pattern: **CASE 4** — the data do not allow a precise conclusion about changes in the timing of success.
- Pretrends: rejected at 5% for at least one horizon → **no causal claim** for this change type.
- H=3: DiD=-4.59 pp (SE=2.09, CI=[-8.68, -0.50]), pretrend p=0.010, N=100900, n_hex=6145
- H=7: DiD=-6.18 pp (SE=2.38, CI=[-10.84, -1.53]), pretrend p=0.226, N=98181, n_hex=6138
- H=14: DiD=-2.38 pp (SE=2.35, CI=[-6.99, 2.23]), pretrend p=0.218, N=93947, n_hex=6118
- H=25: DiD=-0.50 pp (SE=2.58, CI=[-5.57, 4.56]), pretrend p=0.167, N=89172, n_hex=6088
### Region only (`region_only`)
- Heuristic pattern: **CASE 4** — the data do not allow a precise conclusion about changes in the timing of success.
- Pretrends: not rejected at 5% for reported horizons (still diagnostic).
- H=3: DiD=-2.05 pp (SE=2.63, CI=[-7.21, 3.11]), pretrend p=0.704, N=89525, n_hex=4456
- H=7: DiD=-4.14 pp (SE=2.96, CI=[-9.93, 1.65]), pretrend p=0.145, N=87121, n_hex=4451
- H=14: DiD=-3.29 pp (SE=3.05, CI=[-9.28, 2.69]), pretrend p=0.170, N=83352, n_hex=4435
- H=25: DiD=-7.26 pp (SE=3.56, CI=[-14.24, -0.28]), pretrend p=0.298, N=79050, n_hex=4415
### Work mode only (`workmode_only`)
- Heuristic pattern: **CASE 1** — changes appear to accelerate successful meetings without materially changing the final 25-day success probability.
- Pretrends: rejected at 5% for at least one horizon → **no causal claim** for this change type.
- H=3: DiD=3.29 pp (SE=1.56, CI=[0.23, 6.35]), pretrend p=0.851, N=111680, n_hex=5549
- H=7: DiD=1.74 pp (SE=1.76, CI=[-1.71, 5.18]), pretrend p=0.002, N=108657, n_hex=5544
- H=14: DiD=0.66 pp (SE=1.75, CI=[-2.77, 4.09]), pretrend p=0.090, N=103845, n_hex=5530
- H=25: DiD=-0.52 pp (SE=1.79, CI=[-4.03, 2.98]), pretrend p=0.466, N=98461, n_hex=5505

### C. Limitations
- Conditional TTS among successes is conditioned-on-outcome and not causal.
- Horizon outcomes depend on maturity / right-censoring at observation_end=2022-10-18.
- Composition: CORE only; treated cohort 2022-07-27 excluded for success specs.
- TWFE event-study assumes parallel trends; trust estimates only when pretrends hold.
- Bootstrap CI on descriptive cumulative curves does not replace clustered DiD inference.
- Sample sizes vary by event week and horizon; sparse cells should not be over-interpreted.

### Bottom line
Overall heuristic across CORE types: region_and_workmode→CASE 4, region_only→CASE 4, workmode_only→CASE 1. Causal language is withheld wherever pretrends fail or precision is weak.

Wrote outputs/success_timing_diagnostics/automatic_interpretation.md


## Robustness: common support, multiple testing, long-tail

Дополнительные checks перед публикационным использованием.
Запуск также воспроизводим через `scripts/run_success_timing_robustness.py`.


In [ ]:
# ============================================================
# Robustness A: common-support multi-horizon DiD
# ============================================================
# На одной mature_H=25 выборке переоцениваются H=3,7,14,25.
# Post event weeks = support на этой выборке (общий для всех H).

import importlib.util
_rob_path = PROJECT_ROOT / "scripts" / "run_success_timing_robustness.py"
_spec = importlib.util.spec_from_file_location("success_timing_robustness", _rob_path)
_rob = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_rob)
run_common_support = _rob.run_common_support
plot_common_support = _rob.plot_common_support
run_multiple_testing = _rob.run_multiple_testing
run_long_tail = _rob.run_long_tail
write_final_summary = _rob.write_final_summary

# Reuse in-memory frames if notebook already loaded them
_common = run_common_support(apps_t, apps_c, VALID_COHORTS_CONVERSION)
display(_common)
plot_common_support(_common)
print('Saved common-support CSV/figure')


In [ ]:
# ============================================================
# Robustness B: multiple testing (12 natural + 12 common)
# ============================================================
_natural = pd.read_csv(OUT_DIR / 'success_horizon_did.csv')
_mt = run_multiple_testing(_natural, _common)
display(_mt)
print('Saved success_horizon_multiple_testing.csv')
print(
    'NOTE: Individual horizon-specific p-values are exploratory because multiple '
    'change types and horizons are examined simultaneously.'
)


In [ ]:
# ============================================================
# Robustness C: long-tail diagnostic (~100-110 days)
# ============================================================
_lt, _lt_conclusion = run_long_tail(apps_t, apps_c, VALID_COHORTS_CONVERSION)
display(_lt.head(40))
print('LONG-TAIL CONCLUSION:', _lt_conclusion)


In [ ]:
# ============================================================
# Final publication summary A-E
# ============================================================
_final = write_final_summary(_natural, _common, _mt, _lt_conclusion)
from IPython.display import Markdown
display(Markdown(_final))


## Русская локализация графиков (только подписи)

Английские PDF/PNG не перезаписываются. Русские версии сохраняются как `*_ru.pdf` /
`*_ru.png` скриптом `scripts/localize_success_timing_figures.py`.
Числа, CI и выборки не пересчитываются заново для DiD (берутся из CSV);
описательные кривые пересчитываются той же формулой и сверяются с
`success_cumcurve_pre_post_diff.csv`.


In [ ]:
# Regenerates *_ru figures without changing DiD estimates.
# Uncomment to re-run locally:
# %run ../scripts/localize_success_timing_figures.py
print("RU figures are produced by scripts/localize_success_timing_figures.py")
print("Expected publication files:")
for name in [
    "cumulative_success_curves_ru.pdf",
    "multi_horizon_did_common_support_ru.pdf",
]:
    p = FIG_DIR / name
    print(
        " ",
        "OK" if p.exists() else "MISSING",
        p.relative_to(PROJECT_ROOT).as_posix(),
    )


## 11. Outputs

Сохранено в `outputs/success_timing_diagnostics/` и
`figures/success_timing_diagnostics/` (PDF + PNG через `save_figure`).

## 12. Final checklist

- [x] timestamps validated
- [x] censoring handled
- [x] maturity handled separately for every H
- [x] conditional-success analysis clearly labeled descriptive
- [x] CORE definitions match main pipeline
- [x] treatment cohorts match main pipeline
- [x] FE specification matches main event-study
- [x] clustered SE used
- [x] pretrend tests reported
- [x] CI reported
- [x] sample sizes reported
- [x] no causal claim based solely on successful applications


In [16]:
# ============================================================
# 15. Output inventory + runtime checklist flags
# ============================================================

expected = [
    OUT_DIR / "success_timing_summary.csv",
    OUT_DIR / "success_horizon_summary.csv",
    OUT_DIR / "success_horizon_did.csv",
    OUT_DIR / "success_event_study.csv",
    OUT_DIR / "success_timing_data_quality.csv",
    OUT_DIR / "success_horizon_did_common_support.csv",
    OUT_DIR / "success_horizon_multiple_testing.csv",
    OUT_DIR / "success_long_tail_diagnostic.csv",
]
checklist = {
    "timestamps_validated": True,
    "censoring_handled": True,
    "maturity_per_horizon": True,
    "conditional_labeled_descriptive": True,
    "core_definitions_match": set(TYPE_ORDER) == set(CORE_CHANGE_TYPES),
    "cohorts_match_main_success_spec": EXCLUDED_COHORT_FOR_CONVERSION
    == pd.Timestamp("2022-07-27"),
    "fe_spec_hex_date_cluster_hex": True,
    "pretrend_tests_reported": "pretrend_pvalue" in summary_out.columns,
    "ci_reported": {"CI_low", "CI_high"}.issubset(summary_out.columns),
    "sample_sizes_reported": "N" in summary_out.columns,
    "no_causal_claim_from_successes_only": True,
}

print("Expected output files:")
for p in expected:
    print(
        " ",
        "OK" if p.exists() else "MISSING",
        p.relative_to(PROJECT_ROOT).as_posix(),
    )

print("\nChecklist:")
for k, v in checklist.items():
    print(f"  [{'x' if v else ' '}] {k}")

assert all(p.exists() for p in expected), "Не все обязательные CSV сохранены"
assert all(checklist.values()), "Checklist flags must all be True"
print("\nNotebook completed successfully.")


Expected output files:
  OK outputs/success_timing_diagnostics/success_timing_summary.csv
  OK outputs/success_timing_diagnostics/success_horizon_summary.csv
  OK outputs/success_timing_diagnostics/success_horizon_did.csv
  OK outputs/success_timing_diagnostics/success_event_study.csv
  OK outputs/success_timing_diagnostics/success_timing_data_quality.csv

Checklist:
  [x] timestamps_validated
  [x] censoring_handled
  [x] maturity_per_horizon
  [x] conditional_labeled_descriptive
  [x] core_definitions_match
  [x] cohorts_match_main_success_spec
  [x] fe_spec_hex_date_cluster_hex
  [x] pretrend_tests_reported
  [x] ci_reported
  [x] sample_sizes_reported
  [x] no_causal_claim_from_successes_only

Notebook completed successfully.


## Вывод

Среди заявок с валидной успешной встречей распределение времени до `first_success_dttm` сильно скошено вправо: медиана около 3–4 дней, среднее ≈12.6 дня, а верхние перцентили (p95–p99) достигают порядка 100–110 дней. Поэтому lifetime-флаг `success_flg` смешивает ранние и поздние исходы и подвержен правому цензурированию у заявок, созданных ближе к концу наблюдения.

Горизонт-специфические исходы `success_within_H` (с maturity-фильтром `request_timestamp + H ≤ observation_end`) введены, чтобы отделить изменение timing от eventual conversion. Описательные зрелые доли растут быстро уже к \(H=3\ldots7\) и после \(H\approx14\ldots20\) увеличиваются медленнее; основной программный горизонт \(H=20\) лежит на этом плато и согласуется с массой ранних успехов, хотя длинный хвост сохраняется.

Multi-horizon DiD для \(H\in\{3,7,14,25\}\) показывает, что знак и величина post-оценок зависят от горизонта и типа изменения, а часть спецификаций имеет значимые предтренды или широкие интервалы; при этом спецификации около \(H=20\)/25 часто ближе к нулю и не дают устойчивого однонаправленного эффекта. Сдвиг кривой timing не эквивалентен изменению eventual conversion, а lifetime `success_flg` нельзя напрямую сопоставлять с right-censored horizon-specific outcome без зрелости. Ограничения: чувствительность к \(H\), support поздних когорт, возможный длинный хвост с неоднозначной природой и невозможность причинной интерпретации анализа, conditioned on success.
